# Product Scraping Agent — Single URL Run

This notebook runs the URL → product-only evidence artifact pipeline. It supports the v1.1.7 source-alignment contract, where the requested retailer/country can be different from the actual URL being scraped.

Search/discovery is **not** performed here. If you already have SerpAPI/AI Mode/snippet evidence, pass it as optional upstream evidence.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_PATH = PROJECT_ROOT / 'src'

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('SRC_PATH:', SRC_PATH)
print('SRC exists:', SRC_PATH.exists())


In [ ]:
from dotenv import load_dotenv

env_path = PROJECT_ROOT / '.env'
if env_path.exists():
    load_dotenv(env_path)
    print('Loaded .env:', env_path)
else:
    print('No .env found. Copy .env.example to .env and fill LLM/browser settings if needed.')


In [ ]:
from product_scraping_agent import ProductScrapingAgent, ScrapeRequest

print('Import successful')


## Configure request

Use `product_url` as the actual evidence source URL. The requested retailer/country can be different from that URL when you are using a fallback source.

Recommended `source_url_role` values:

| Role | When to use |
|---|---|
| `primary_requested_retailer` | URL is from the requested retailer/country |
| `same_retailer_different_country` | Same retailer, different country domain |
| `alternate_retailer_same_country` | Different retailer, same country |
| `alternate_retailer_different_country` | Different retailer and different country |
| `marketplace_fallback` | Marketplace fallback source |
| `global_fallback` | Global/other source used only for product facts |
| `unknown` | Source relationship not known |


In [ ]:
request = ScrapeRequest(
    # Required: actual URL to scrape
    product_url='https://www.orellfuessli.ch/shop/home/artikeldetails/A1079084180',

    # Optional product identity hints
    main_text='PUT_PRODUCT_MAIN_TEXT_HERE',
    ean='',

    # Original requested market context from your input/discovery stage
    requested_retailer_name='PUT_REQUESTED_RETAILER_HERE',
    requested_country_code='CO',

    # Actual evidence source context for the URL above
    source_retailer_name='PUT_SOURCE_RETAILER_HERE',
    source_country_code='CH',
    source_url_role='global_fallback',

    # Optional upstream evidence from search/discovery. The scraper does not search.
    upstream_ai_evidence='',
    candidate_snippets=[],
    search_evidence=[],

    output_root=PROJECT_ROOT / 'data' / 'scraped',
    max_images=30,
    vision_max=12,
    max_agent_iterations=2,
    strict_product_only=True,
    write_raw_debug=False,
)

request.model_dump()


In [ ]:
agent = ProductScrapingAgent()
result = await agent.scrape(request)
result.model_dump()


In [ ]:
print('Success:', result.success)
print('Quality:', result.artifact_quality)
print('Manual review:', result.requires_manual_review)
print('Output dir:', result.output_dir)
print('Product evidence JSON:', result.product_evidence_json_path)
print('Product evidence MD:', result.product_evidence_md_path)
print('Claims MD:', result.claims_md_path)
print('Vision MD:', result.vision_md_path)
print('Quality report:', result.quality_report_path)
print('Source alignment:', result.source_alignment_report_path)
print('Error:', result.error)


In [ ]:
import json

def read_json(path):
    path = Path(path)
    return json.loads(path.read_text(encoding='utf-8')) if path and path.exists() else {}

quality = read_json(result.quality_report_path)
alignment = read_json(result.source_alignment_report_path)

print('QUALITY REPORT')
print(json.dumps(quality, indent=2, ensure_ascii=False)[:4000])
print('\nSOURCE ALIGNMENT REPORT')
print(json.dumps(alignment, indent=2, ensure_ascii=False)[:4000])


## Expected clean artifact files

The important outputs are:

```text
retailer/
├── source.md                         # product-only source digest
├── product_evidence.json             # structured product-only evidence
├── product_evidence.md               # table-first product evidence
├── claims.md                         # business-level claim dossier
├── vision.md                         # table-first visual evidence
├── metadata.json
├── source_alignment_report.json      # requested context vs actual URL/source
├── evidence_recovery_report.json
├── quality_report.json
├── noise_report.json
├── images/                           # only final relevant product images
├── tables/
└── manifests/
```
